# Pathology Report: Three Reusable Stages
Run on a remote Delta Python kernel. Change the image filename in Stage 1, then run the three stages in order (or choose Run All).

Each stage saves its result to disk. Re-running cells or restarting the kernel reuses matching completed results instead of submitting another GPU job. After a kernel restart, run Stage 1 and Stage 2 again to restore the session; completed work is read from disk.

If MedGemma fails, fix its environment and retry only Stage 2; PathGen is not repeated. Likewise, retry only Stage 3 if MUSK fails. Intermediate logs stay hidden unless a stage fails.

To regenerate a completed stage intentionally, use `force=True`, e.g. `session.medgemma(force=True)`. Re-run the following stages to assemble the updated report. A pending job is reattached rather than duplicated. Run one session at a time.

## 1. PathGen — caption
Default MUSK checkpoint: `../musk/runs/head_slurm_22984898/best.pt`. All paths and model files stay on the server.


In [ ]:
# Change only this image filename.
IMAGE_FILENAME = "TUM-YHTSLLSL.tif"

# Optional: import an existing caption instead of running PathGen again.
# Example: EXISTING_CAPTION = "reports/pathgen_retry1.json"
EXISTING_CAPTION = None

from pathlib import Path
import sys
import importlib

_connect = None
for _parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    for _candidate in (_parent, _parent / "connect"):
        if (_candidate / "staged_runner.py").is_file():
            _connect = _candidate
            break
    if _connect is not None:
        break
if _connect is None:
    raise RuntimeError("Open the remote repository connect folder before running this notebook.")
if str(_connect) not in sys.path:
    sys.path.insert(0, str(_connect))

import notebook_runner
import staged_runner
importlib.reload(notebook_runner)
importlib.reload(staged_runner)
from notebook_runner import Settings
from staged_runner import ReportSession

settings = Settings(connect=_connect)
session = ReportSession(IMAGE_FILENAME, settings)
_existing = (_connect / EXISTING_CAPTION) if EXISTING_CAPTION else None
caption = session.pathgen(existing_caption=_existing)
print(caption["caption"])


## 2. MedGemma — plain-English explanation
Uses the caption from Stage 1.


In [ ]:
# Stage 2 uses the saved PathGen caption. Retry only this cell if MedGemma fails.
explanation = session.medgemma()
print(explanation["simplified_caption"])


## 3. MUSK — five predictions and final report
Displays the explanation followed by five named yes/no/unknown predictions.


In [ ]:
# Stage 3 runs MUSK independently and appends its five predictions.
result = session.musk()
print(result["report"])
